# 03 - Calibration: the YAML declares the parameters, IPOPT finds them

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andoludo/mpctrano/blob/main/tutorials/03_calibration_from_yaml.ipynb)

**Learning objectives**

1. Declare the parameters to calibrate in the Trano YAML itself (`{value, parameter: true, min, max}`), keep it
   the single source of truth, and see which parameters of the CasADi model each field moves.
2. Calibrate the **closed-loop** model (the controllers of the YAML inside, no control input) on monitoring data
   a real building would give: a noisy air temperature and an electricity meter, no valve positions.
3. Read how IPOPT solves it: the three fields as decision variables, the closed-loop model simulated with CVODES
   inside a weighted least squares with priors, the gradient from the adjoint sensitivities of the integrator.
4. Compare the model before and after: the temperatures and the energy on the training days and on two
   validation days, and the recovered values against the true ones.

The true building, its data and the calibration come from `tutorials/cache/calibration/` (`make cache`); the
calibration is solved live when the notebook does not run in quick mode (a few minutes).

**Runtimes** (this container): one regeneration of the model by Trano and rumoca about 15 s (one per declared
parameter, for the sensitivities); the calibration of three days of hourly data, 57 states, two to five minutes.

In [1]:
import sys

if "google.colab" in sys.modules:  # Colab: fetch the tutorials, install mpctrano and rumoca (a no-op elsewhere)
    import subprocess

    subprocess.run(  # a fixed command; the clone is skipped when it exists
        ["git", "clone", "-q", "--depth", "1", "https://github.com/andoludo/mpctrano", "/content/mpctrano"], check=False
    )
    sys.path.insert(0, "/content/mpctrano")
    from tutorials import _colab

    _colab.setup()

In [2]:
import pandas as pd
from IPython.display import display

from mpctrano import CalibrationSettings, Calibrator
from mpctrano.calibration import evaluate, simulate_fit
from mpctrano.notebook import quick_mode, timed
from mpctrano.plot import plot_fit
from tutorials import _calibration as study

pd.set_option("display.width", 200)
print(f"quick mode: {quick_mode()}")

quick mode: False


## 1. The parameters, declared in the YAML

`tests/models/single_zone_calibration.yaml` is the heat pump house of tutorial 01 with its cavity wall spelled out
and three fields annotated: the conductivity of the insulation, its thickness and the infiltration rate.
`CalibrationSpec` reads them by path (the convention of `trano.runtime`) and writes the flat copy Trano reads.

In [3]:
spec = study.spec()
text = study.YAML.read_text()
print(text[: text.index("    id: SPACE:001")])
display(pd.DataFrame([parameter.model_dump(by_alias=True) for parameter in spec.parameters]).set_index("path"))

# The single-zone heat pump house with calibration parameters declared in the YAML: a numeric field written as
# {value, parameter: true, min, max} is calibrated within its bounds (mpctrano.yaml_parameters); Trano reads the
# flat copy mpctrano writes. The cavity wall is spelled out so that its insulation can be calibrated.
default: !include_default
material:
  - id: ROCKWOOL:CAL
    density: 60.0
    longwave_emissivity: 0.9
    shortwave_emissivity: 0.6
    specific_heat_capacity: 840.0
    thermal_conductivity: {value: 0.04, parameter: true, min: 0.025, max: 0.08}
constructions:
  - id: CAVITYWALL:CAL
    layers:
      - material: BRICK:001
        thickness: 0.08
      - material: ROCKWOOL:CAL
        thickness: {value: 0.1, parameter: true, min: 0.04, max: 0.2}
      - material: BRICKHOLLOW:001
        thickness: 0.14
      - material: GYPSUM:001
        thickness: 0.015
spaces:
  - occupancy:
      parameters:
        occupancy: 3600*{7, 22}
        gain: "[40; 75; 40]"
        h

,value,min,max
path,,,
material[ROCKWOOL:CAL].thermal_conductivity,0.04,0.025,0.08
constructions[CAVITYWALL:CAL].layers[1].thickness,0.10,0.040,0.20
spaces[SPACE:001].parameters.ach,0.30,0.050,1.50


Which parameters of the CasADi model a field moves is **measured**: the model is regenerated with the field raised
by 20 % and the exponent of every parameter that moved is read. The conductivity moves the conductance of the
insulation layer of the three walls (exponent 1), the thickness moves it with -1 and the capacities of the
neighbouring nodes a little, the infiltration rate is the `ACH` of the zone.

The table also says what *not* to calibrate together: the conductivity and the thickness enter the model through
the same conductance (`k / x`, exponents +1 and -1), the capacities they move a little apart; no data on the air
temperature tells them apart. The study keeps the thickness as drawn (a drawing is a measurement) and
calibrates the conductivity and the infiltration rate.

In [4]:
for group in study.declared_groups():
    moved = {
        name: round(exponent, 3)
        for name, exponent in group.exponents.items()
        if not name.startswith(("externalwall_1", "externalwall_2"))
    }
    print(f"{group.name}: value {group.value:g} in [{group.lower:g}, {group.upper:g}] -> {moved} (and the other walls)")
groups = study.groups()
print("calibrated:", [group.name for group in groups])

material[ROCKWOOL:CAL].thermal_conductivity: value 0.04 in [0.025, 0.08] -> {'externalwall_0.constructionType.k[2]': 1.0, 'externalwall_0.k[3]': 1.0, 'externalwall_0.k[4]': 1.0, 'externalwall_0.G[3]': 1.0, 'externalwall_0.G[4]': 1.0} (and the other walls)
constructions[CAVITYWALL:CAL].layers[1].thickness: value 0.1 in [0.04, 0.2] -> {'externalwall_0.constructionType.x[2]': 1.0, 'externalwall_0.x[3]': 1.0, 'externalwall_0.x[4]': 1.0, 'externalwall_0.C[3]': 0.043, 'externalwall_0.C[4]': 1.0, 'externalwall_0.C[5]': 0.073, 'externalwall_0.G[3]': -1.0, 'externalwall_0.G[4]': -1.0} (and the other walls)
spaces[SPACE:001].parameters.ach: value 0.3 in [0.05, 1.5] -> {'space_001.ACH': 1.0} (and the other walls)
calibrated: ['material[ROCKWOOL:CAL].thermal_conductivity', 'spaces[SPACE:001].parameters.ach']


## 2. The true building and its monitoring data

The *true* building is the same YAML with other values of the three parameters: a worse insulation, thinner than
drawn, and a leakier house. It is run by its own controllers (the closed-loop `FastModel`: thermostat, heating
curve, buffer control) over five cold days of January, after two days of spin-up; the monitoring system records
the weather, the air temperature with 0.1 K of noise and the electricity meter with 20 Wh of noise. It does
**not** record the valves or the compressor: the model being calibrated has its controllers inside.

In [5]:
model = study.model()
print(
    "states",
    model.n_states,
    "| controls",
    model.control_names,
    "| measured",
    model.comfort_states,
    "| meters",
    list(model.meters),
)
truth = pd.Series(spec.nominal, name="assumed (YAML)").to_frame()
truth["true value"] = pd.Series(study.TRUTH).reindex(truth.index).fillna(truth["assumed (YAML)"])
display(truth)
data, true_states = study.recorded()
training, validation = study.split(data)
print(f"{len(training) - 1} training hours from {training.index[0]}, {len(validation) - 1} validation hours")
display(data[[*model.comfort_states, *model.meters]].describe().round(2).T)

states 57 | controls [] | measured ['space_001.TAir'] | meters ['boiler_001.EEle']


,assumed (YAML),true value
material[ROCKWOOL:CAL].thermal_conductivity,0.04,0.06
constructions[CAVITYWALL:CAL].layers[1].thickness,0.10,0.10
spaces[SPACE:001].parameters.ach,0.30,0.60


72 training hours from 2023-01-09 00:00:00, 48 validation hours


,count,mean,std,min,25%,50%,75%,max
space_001.TAir,121.0,290.08,0.65,288.70,289.54,290.18,290.56,291.26
boiler_001.EEle,121.0,37.06,13.90,12.07,24.99,38.95,49.40,58.19


## 3. Before: the model with the values of the YAML

An open-loop simulation of the model from the first measurement, with the assumed parameters: the house loses
less heat than the true one, so it stays warmer and draws less electricity.

In [6]:
before = simulate_fit(model, training)
metrics_before, _ = evaluate(model, training)
display(plot_fit(before, "before calibration: training days (dotted: measured)"))
display(metrics_before.frame().round(3))

,rmse,mae,max_error,r2
space_001,1.379,1.271,1.891,-3.655


## 4. How IPOPT calibrates it

The calibration is one nonlinear programme over the whole training period, solved by **single shooting**
(`CalibrationSettings.method`, chosen automatically for a closed-loop model):

* **decision variables**: one per declared parameter, `log(field / value)`, bounded by `log(min / value)` and
  `log(max / value)`; nothing else. The parameters of the model are expressions of the fields
  (`p = p0 * (field / value) ** e`, the exponents measured above; a conductivity and a thickness both move the
  conductance of the insulation layer, their factors compound).
* **the simulation inside the objective**: CVODES integrates the closed-loop model hour by hour, with the
  parameters of the candidate, from the start values of the Modelica model through two days of spin-up before
  the data (so that the walls, the water and the controller states at the start are those of the candidate, not
  of the nominal model) and then over the training days with their weather.
* **the gradient**: central differences of the objective with a step of 5 % of a field, 2n + 1 simulations of a
  second each; IPOPT's Hessian is a quasi-Newton (L-BFGS) one and it takes its full steps. A thermostat that
  flips one integration step earlier makes the objective jumpy at the scale of a tenth of a percent of a field:
  the adjoint sensitivities of the integrator and small finite differences read that noise and stall IPOPT
  short of the optimum, a wide step reads the slope of the valley through it and a sufficient-decrease test
  would refuse steps for the same jumps.
* **objective**: the squared errors of the simulated air temperature against the measurements (weighted by the
  sensor noise), of the energy metered over each hour against the meter (weighted by the meter noise), and the
  squared deviations of the fields from their prior (the YAML).

The models with control inputs (the MPC models of the other tutorials) are calibrated on their Radau
collocation instead, the states as decision variables and the dynamics as constraints: well conditioned over
long periods and poor initial parameters. Not here: the thermostat, the hysteresis of the heat pump controller
and the valve limits make the collocated dynamics of a closed-loop model non-smooth, and IPOPT crawls for
hundreds of iterations without converging; a simulation follows these switches in under a second, and three
parameters need a handful of iterations. The log below is IPOPT's own (one line per iteration: the objective,
the dual infeasibility, the step).

In [7]:
if quick_mode():
    calibrated, result = study.calibrate()
    print("cached result:", result.status, "in", result.iterations, "iterations,", f"{result.solve_time:.0f} s")
else:
    with timed("calibration"):
        settings = CalibrationSettings(groups=groups, verbose=True, max_iterations=60)
        calibrated, result = Calibrator(model, settings).fit(training, validation)
print(
    f"{result.method}: {result.variables} decision variables, {result.iterations} iterations, "
    f"{result.solve_time:.0f} s, {result.status}"
)

This is Ipopt version 3.14.19, running with linear solver spral.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        0



Total number of variables............................:        2
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        2
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0



iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  1.4099103e+04 0.00e+00 1.00e+02   0.0 0.00e+00    -  0.00e+00 0.00e+00   0


   1  7.7341599e+04 0.00e+00 8.90e+02   1.6 4.46e+01    -  8.45e-01 3.57e-02h  1


   2  1.3483015e+04 0.00e+00 2.70e+02   1.1 4.57e-01    -  7.50e-02 1.00e+00f  1


...

  11  9.0826975e+01 0.00e+00 2.25e-01  -9.1 3.82e-02    -  1.00e+00 1.00e+00h  1


  12  9.0433672e+01 0.00e+00 4.50e-01  -9.0 2.62e-02    -  1.00e+00 1.00e+00f  1


  13  9.0462485e+01 0.00e+00 2.15e-01  -9.0 8.74e-04    -  1.00e+00 1.00e+00h  1


  14  9.0567557e+01 0.00e+00 6.61e-02  -9.0 1.23e-03    -  1.00e+00 1.00e+00h  1

Number of Iterations....: 14

                                   (scaled)                 (unscaled)
Objective...............:   3.8448097069631770e-01    9.0567556557836213e+01
Dual infeasibility......:   6.6091914483216024e-02    1.5568477139801574e+01
Constraint violation....:   0.0000000000000000e+00    0.0000000000000000e+00
Variable bound violation:   0.0000000000000000e+00    0.0000000000000000e+00
Complementarity.........:   1.0835244551212606e-09    2.5523281995795604e-07
Overall NLP error.......:   6.6091914483216024e-02    1.5568477139801574e+01


Number of objective function evaluations             = 15
Number of objective gradient evaluations             = 15
Number of equality constraint evaluations            = 0
Number of inequality constraint evaluations          = 0
Number of equality constraint Jacobian evaluations   = 0
Number of inequality constraint Jacobian evaluations = 0
Number of

calibration: 159.4 s
shooting: 2 decision variables, 14 iterations, 153 s, Solved_To_Acceptable_Level


## 5. After: the fit on the training and validation days, and what the two fields became

In [8]:
recovered = pd.DataFrame({"assumed (YAML)": spec.nominal, "calibrated": result.factors, "true": truth["true value"]})
recovered["error %"] = 100 * (recovered["calibrated"] - recovered["true"]) / recovered["true"]
display(recovered.round(3))
display(result.metrics().round(3))

,assumed (YAML),calibrated,true,error %
constructions[CAVITYWALL:CAL].layers[1].thickness,0.10,NaN,0.10,NaN
material[ROCKWOOL:CAL].thermal_conductivity,0.04,0.055,0.06,-8.888
spaces[SPACE:001].parameters.ach,0.30,0.616,0.60,2.712


,rmse,mae
training (Trano),1.379,1.271
training (calibrated),0.096,0.077
validation (Trano),1.127,1.070
validation (calibrated),0.098,0.083


In [9]:
x0 = study.initial_state(result)
after = simulate_fit(calibrated, training, x0)
display(plot_fit(after, "after calibration: training days (dotted: measured)"))
after_validation = simulate_fit(calibrated, validation)
before_validation = simulate_fit(model, validation)
display(plot_fit(after_validation, "after calibration: validation days (dotted: measured)"))

## 6. Identifiability: what the data can and cannot tell apart

The calibrated model reproduces the air temperature and the electricity of the true building on days it has not
seen, but the two fields did not land on their true values: a leakier house and a worse insulation are both a
loss proportional to the indoor-outdoor temperature difference. The data of a thermostat-controlled house fix the
**total** loss; how it splits between the walls and the infiltration is a flat valley of the objective, and the
prior (the YAML) and the first steps of IPOPT decide where along it the fit stops.

What a measurement can add: a blower-door test gives the infiltration rate. With the YAML regenerated at that
rate and the conductivity as the only declared parameter, the same fit recovers it.

In [10]:
with timed("calibration with the infiltration measured"):
    calibrated_known, result_known = study.calibrate(known_infiltration=True)
print(
    f"{result_known.method}: {result_known.variables} decision variable, {result_known.iterations} iterations, "
    f"{result_known.solve_time:.0f} s, {result_known.status}"
)
known = pd.DataFrame(
    {
        "assumed (YAML)": {study.CONDUCTIVITY: spec.nominal[study.CONDUCTIVITY]},
        "calibrated": result_known.factors,
        "true": {study.CONDUCTIVITY: study.TRUTH[study.CONDUCTIVITY]},
    }
)
known["error %"] = 100 * (known["calibrated"] - known["true"]) / known["true"]
display(known.round(4))
display(result_known.metrics().round(3))

calibration with the infiltration measured: 7.5 s
shooting: 1 decision variable, 15 iterations, 107 s, Solved_To_Acceptable_Level


,assumed (YAML),calibrated,true,error %
material[ROCKWOOL:CAL].thermal_conductivity,0.04,0.0594,0.06,-1.0411


,rmse,mae
training (Trano),0.324,0.303
training (calibrated),0.094,0.076
validation (Trano),0.284,0.247
validation (calibrated),0.098,0.083


## Key takeaways

* **The YAML is the specification of the calibration too**: three fields carry `{value, parameter: true, min, max}`,
  `CalibrationSpec` writes the flat copy Trano reads and measures which parameters of the CasADi model each field
  moves; the fit adjusts the fields within their bounds and the model follows.
* **A closed-loop model is calibrated by simulation, not collocation**: its thermostat and hysteresis make the
  collocated problem non-smooth; the single-shooting fit (two decision variables, CVODES inside the objective,
  wide central differences) brings the air-temperature error from 1.38 K to 0.10 K on the training days and
  from 1.13 K to 0.10 K on two unseen days, with the energy use to match.
* **The data decide what is identifiable**: the free fit lands at a conductivity of 0.055 and 0.62 air changes
  per hour (true: 0.060 and 0.60) - the right total loss, split along a flat valley the objective barely
  feels; with the infiltration measured, the conductivity comes out at 0.0594. Declare in the YAML what a
  measurement cannot give, fix the rest.
* **The remaining 0.1 K is the switching**: a thermostat that flips one integration step earlier moves the hourly
  sample; the fitted model has the right parameters but not the exact switching times, which is what IPOPT's
  jumpy log in section 4 shows.

## Further reading

* `docs/calibration.md` for the groups and the problem; `mpctrano.yaml_parameters` for the annotations.
* Trano's `runtime` package addresses the same YAML fields by path for the Buildings models compiled with
  OpenModelica (`trano.runtime.specification`): the two conventions agree.